# Caso Práctico: Redes Neuronales Artificiales (RNA)

**Curso:** Introducción al Aprendizaje Profundo - Primera entrega

**Equipo:** Juan José Herrera y Santiago Rodríguez

**Repositorio (más información):** https://github.com/JuanJoseHQ/introduccion_aprendizaje_profundo


## Conjunto de datos: Detección de transacciones bancarias fraudulentas

### Descripción
Conjunto de datos con transacciones realizadas con tarjetas de crédito por clientes europeos durante dos días de septiembre de 2013. Incluye 284.807 transacciones, de las cuales solo 492 son fraudulentas (0,172%), por lo que está muy desequilibrado.

Todas las variables de entrada son numéricas. Las características `V1` a `V28` son componentes principales obtenidos con PCA; por motivos de confidencialidad no se publican las variables originales. Las únicas que no se transformaron son:

- `Time`: segundos transcurridos entre cada transacción y la primera del conjunto.
- `Amount`: importe de la transacción.
- `Class`: variable objetivo, vale 1 si la transacción es fraudulenta y 0 en caso contrario.

### Descarga de los ficheros de datos
https://www.kaggle.com/mlg-ulb/creditcardfraud#creditcard.csv

### Referencias adicionales sobre el conjunto de datos
_The dataset has been collected and analysed during a research collaboration of Worldline and the Machine Learning Group (http://mlg.ulb.ac.be) of ULB (Université Libre de Bruxelles) on big data mining and fraud detection.
More details on current and past projects on related topics are available on https://www.researchgate.net/project/Fraud-detection-5 and the page of the DefeatFraud project._

## Imports

In [35]:
%matplotlib inline
import pandas as pd
from sklearn.model_selection import train_test_split

## Funciones auxiliares

In [36]:
# Construcción de una función que realice el particionado completo
def train_val_test_split(df, rstate=42, shuffle=True, stratify=None):
    strat = df[stratify] if stratify else None
    train_set, test_set = train_test_split(
        df, test_size=0.4, random_state=rstate, shuffle=shuffle, stratify=strat)
    strat = test_set[stratify] if stratify else None
    val_set, test_set = train_test_split(
        test_set, test_size=0.5, random_state=rstate, shuffle=shuffle, stratify=strat)
    return (train_set, val_set, test_set)

In [37]:
def remove_labels(df, label_name):
    X = df.drop(label_name, axis=1)
    y = df[label_name].copy()
    return (X, y)

## 1. Lectura del conjunto de datos

In [38]:
df = pd.read_csv("../data/creditcard.csv")

## 2. Visualización del conjunto de datos

In [39]:
df.head(10)

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0
5,2.0,-0.425966,0.960523,1.141109,-0.168252,0.420987,-0.029728,0.476201,0.260314,-0.568671,...,-0.208254,-0.559825,-0.026398,-0.371427,-0.232794,0.105915,0.253844,0.081080,3.67,0
6,4.0,1.229658,0.141004,0.045371,1.202613,0.191881,0.272708,-0.005159,0.081213,0.464960,...,-0.167716,-0.270710,-0.154104,-0.780055,0.750137,-0.257237,0.034507,0.005168,4.99,0
7,7.0,-0.644269,1.417964,1.074380,-0.492199,0.948934,0.428118,1.120631,-3.807864,0.615375,...,1.943465,-1.015455,0.057504,-0.649709,-0.415267,-0.051634,-1.206921,-1.085339,40.80,0
8,7.0,-0.894286,0.286157,-0.113192,-0.271526,2.669599,3.721818,0.370145,0.851084,-0.392048,...,-0.073425,-0.268092,-0.204233,1.011592,0.373205,-0.384157,0.011747,0.142404,93.20,0
9,9.0,-0.338262,1.119593,1.044367,-0.222187,0.499361,-0.246761,0.651583,0.069539,-0.736727,...,-0.246914,-0.633753,-0.120794,-0.385050,-0.069733,0.094199,0.246219,0.083076,3.68,0


In [40]:
print("Número de características:", len(df.columns))
print("Longitud del conjunto de datos:", len(df))

Número de características: 31
Longitud del conjunto de datos: 284807


In [41]:
# 492 transacciones fraudulentas, 284315 transacciones legitimas
# El conjunto de datos se encuntra desequilabrado
df["Class"].value_counts()

Class
0    284315
1       492
Name: count, dtype: int64

In [42]:
# Visualizamos los tipos de cada uno de los atributos
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 284807 entries, 0 to 284806
Data columns (total 31 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   Time    284807 non-null  float64
 1   V1      284807 non-null  float64
 2   V2      284807 non-null  float64
 3   V3      284807 non-null  float64
 4   V4      284807 non-null  float64
 5   V5      284807 non-null  float64
 6   V6      284807 non-null  float64
 7   V7      284807 non-null  float64
 8   V8      284807 non-null  float64
 9   V9      284807 non-null  float64
 10  V10     284807 non-null  float64
 11  V11     284807 non-null  float64
 12  V12     284807 non-null  float64
 13  V13     284807 non-null  float64
 14  V14     284807 non-null  float64
 15  V15     284807 non-null  float64
 16  V16     284807 non-null  float64
 17  V17     284807 non-null  float64
 18  V18     284807 non-null  float64
 19  V19     284807 non-null  float64
 20  V20     284807 non-null  float64
 21  V21     28

In [43]:
# Comprobamos si alguna columna tiene valores nulos
df.isna().any()

Time      False
V1        False
V2        False
V3        False
V4        False
V5        False
V6        False
V7        False
V8        False
V9        False
V10       False
V11       False
V12       False
V13       False
V14       False
V15       False
V16       False
V17       False
V18       False
V19       False
V20       False
V21       False
V22       False
V23       False
V24       False
V25       False
V26       False
V27       False
V28       False
Amount    False
Class     False
dtype: bool

In [44]:
df.describe()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
count,284807.000000,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,...,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,2.848070e+05,284807.000000,284807.000000
mean,94813.859575,1.168375e-15,3.416908e-16,-1.379537e-15,2.074095e-15,9.604066e-16,1.487313e-15,-5.556467e-16,1.213481e-16,-2.406331e-15,...,1.654067e-16,-3.568593e-16,2.578648e-16,4.473266e-15,5.340915e-16,1.683437e-15,-3.660091e-16,-1.227390e-16,88.349619,0.001727
std,47488.145955,1.958696e+00,1.651309e+00,1.516255e+00,1.415869e+00,1.380247e+00,1.332271e+00,1.237094e+00,1.194353e+00,1.098632e+00,...,7.345240e-01,7.257016e-01,6.244603e-01,6.056471e-01,5.212781e-01,4.822270e-01,4.036325e-01,3.300833e-01,250.120109,0.041527
min,0.000000,-5.640751e+01,-7.271573e+01,-4.832559e+01,-5.683171e+00,-1.137433e+02,-2.616051e+01,-4.355724e+01,-7.321672e+01,-1.343407e+01,...,-3.483038e+01,-1.093314e+01,-4.480774e+01,-2.836627e+00,-1.029540e+01,-2.604551e+00,-2.256568e+01,-1.543008e+01,0.000000,0.000000
25%,54201.500000,-9.203734e-01,-5.985499e-01,-8.903648e-01,-8.486401e-01,-6.915971e-01,-7.682956e-01,-5.540759e-01,-2.086297e-01,-6.430976e-01,...,-2.283949e-01,-5.423504e-01,-1.618463e-01,-3.545861e-01,-3.171451e-01,-3.269839e-01,-7.083953e-02,-5.295979e-02,5.600000,0.000000
50%,84692.000000,1.810880e-02,6.548556e-02,1.798463e-01,-1.984653e-02,-5.433583e-02,-2.741871e-01,4.010308e-02,2.235804e-02,-5.142873e-02,...,-2.945017e-02,6.781943e-03,-1.119293e-02,4.097606e-02,1.659350e-02,-5.213911e-02,1.342146e-03,1.124383e-02,22.000000,0.000000
75%,139320.500000,1.315642e+00,8.037239e-01,1.027196e+00,7.433413e-01,6.119264e-01,3.985649e-01,5.704361e-01,3.273459e-01,5.971390e-01,...,1.863772e-01,5.285536e-01,1.476421e-01,4.395266e-01,3.507156e-01,2.409522e-01,9.104512e-02,7.827995e-02,77.165000,0.000000
max,172792.000000,2.454930e+00,2.205773e+01,9.382558e+00,1.687534e+01,3.480167e+01,7.330163e+01,1.205895e+02,2.000721e+01,1.559499e+01,...,2.720284e+01,1.050309e+01,2.252841e+01,4.584549e+00,7.519589e+00,3.517346e+00,3.161220e+01,3.384781e+01,25691.160000,1.000000


## 3. Preparación del conjunto de datos

Para este tipo de algoritmos es importante que todos los datos se encuentren en un rango similar, por lo tanto, podemos aplicar una función de escalado o normalización. Otra opción, es eliminar las características que no se encuentran en un rango similar siempre y cuando no sean muy influyentes para la predicción.

In [45]:
df = df.drop(["Time", "Amount"], axis=1)

## 4. División del conjunto de datos

El reparto se hace **estratificado por `Class`**. Como solo el 0,172% de las transacciones son fraudulentas (492 de 284.807), un muestreo puramente aleatorio dejaría una proporción de fraudes distinta en cada subconjunto, y con tan pocos positivos esa variación es apreciable. Al estratificar, entrenamiento, validación y prueba conservan la misma tasa de fraude que el conjunto original, de modo que las métricas de los tres son comparables entre sí.

In [46]:
# Dividimos el conjunto de datos estratificando por la variable objetivo,
# para que los 492 fraudes se repartan de forma proporcional entre los tres subconjuntos
train_set, val_set, test_set = train_val_test_split(df, stratify='Class')

# Comprobamos que la proporción de fraudes se mantiene en los tres subconjuntos
for nombre, subconjunto in [("Entrenamiento", train_set), ("Validación", val_set), ("Prueba", test_set)]:
    fraudes = int(subconjunto["Class"].sum())
    print("%-13s | total: %6d | fraudes: %3d | tasa: %.4f%%"
          % (nombre, len(subconjunto), fraudes, 100 * fraudes / len(subconjunto)))

Entrenamiento | total: 170884 | fraudes: 295 | tasa: 0.1726%
Validación    | total:  56961 | fraudes:  98 | tasa: 0.1720%
Prueba        | total:  56962 | fraudes:  99 | tasa: 0.1738%


In [47]:
X_train, y_train = remove_labels(train_set, 'Class')
X_val, y_val = remove_labels(val_set, 'Class')
X_test, y_test = remove_labels(test_set, 'Class')

## 5. Red Neuronal (MLP) con PyTorch

Entrenamos un perceptrón multicapa sobre los conjuntos ya divididos (`X_train`, `X_val`, `X_test`).

### 5.1. Preparación de los tensores

Antes de entrenar hay que dejar los datos en el formato que espera PyTorch y decidir cómo se le van a ir entregando a la red.

**Escalado.** Las variables `V1`–`V28` provienen de un PCA, que centra los datos pero no iguala sus varianzas: las primeras componentes varían mucho más que las últimas, Sin escalar, esas primeras variables dominarían la magnitud de los gradientes y la red tardaría mucho más en aprovechar el resto. `StandardScaler` deja todas las variables con media 0 y desviación 1.

**Tensores.** PyTorch no trabaja con DataFrames sino con tensores. Se convierten a `float32`, precisión suficiente y estándar en redes neuronales, y las etiquetas se reorganizan con `.view(-1, 1)` para que tengan la misma forma `(N, 1)` que la salida del modelo; si no coincidieran, PyTorch las difundiría silenciosamente y la pérdida se calcularía mal.

**Lotes (*batches*).** La red no se entrena con las 170.884 filas de una sola vez: se recorren en grupos pequeños y, para cada grupo, se calcula la predicción, se mide el error y se actualizan los pesos

**¿Por qué lotes de 512?** El tamaño habitual sería 32 o 64, pero aquí manda el desequilibrio. Con un 0,17% de fraudes, el número esperado de fraudes por lote es `tamaño × 0,0017`:

| Tamaño de lote | Fraudes esperados por lote | Lotes con ≥1 fraude | Lotes por época |
|----------------|----------------------------|---------------------|-----------------|
| 32             | 0,06                       | ~5%                 | 5.340           |
| 128            | 0,22                       | ~20%                | 1.335           |
| **512**        | **0,88**                   | **~59%**            | **334**         |
| 2048           | 3,5                        | ~97%                | 84              |

Con lotes de 32, el 95% de las actualizaciones de pesos se harían sin un solo fraude a la vista: el gradiente repetiría "predice legítima" y las escasas actualizaciones con fraude serían correcciones bruscas en sentido contrario, dando un entrenamiento inestable. Con 512, más de la mitad de los lotes contienen señal de la clase minoritaria y aun así quedan 334 actualizaciones por época.


In [48]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)

torch.manual_seed(42)
np.random.seed(42)

# Escalado: se ajusta solo con entrenamiento
escalador = StandardScaler()
X_train_esc = escalador.fit_transform(X_train)
X_val_esc = escalador.transform(X_val)
X_test_esc = escalador.transform(X_test)

# Conversion a tensores (float32, etiquetas con forma (N, 1))
X_train_t = torch.tensor(X_train_esc, dtype=torch.float32)
y_train_t = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
X_val_t = torch.tensor(X_val_esc, dtype=torch.float32)
y_val_t = torch.tensor(y_val.values, dtype=torch.float32).view(-1, 1)
X_test_t = torch.tensor(X_test_esc, dtype=torch.float32)
y_test_t = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

# DataLoaders (lotes de 512)
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=512, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=512)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=512)

print("Entrenamiento:", X_train_t.shape, y_train_t.shape)
print("Validación   :", X_val_t.shape, y_val_t.shape)
print("Prueba       :", X_test_t.shape, y_test_t.shape)
print("Lotes por época:", len(train_loader))

Entrenamiento: torch.Size([170884, 28]) torch.Size([170884, 1])
Validación   : torch.Size([56961, 28]) torch.Size([56961, 1])
Prueba       : torch.Size([56962, 28]) torch.Size([56962, 1])
Lotes por época: 334


### 5.2. Definición del modelo

**Arquitectura 28 → 128 → 64 → 32 → 1.** La entrada tiene 28 neuronas porque ese es el número de variables tras eliminar `Time` y `Amount`. Las capas ocultas siguen el patrón de embudo habitual

**Dropout 0,2** apaga aleatoriamente el 20% de las neuronas de la capa en cada paso de entrenamiento, forzando a que la predicción no dependa de unas pocas neuronas concretas, es la defensa directa contra memorizar los 295 fraudes.

**pos_weight=5** multiplica por cinco el coste de no detectar un fraude, sin esto, la forma más fácil de minimizar la pérdida sería predecir "legítima" siempre y el modelo colapsaría hacia la clase mayoritaria.

In [ ]:
# MLP: 28 entradas -> 128 -> 64 -> 32 -> 1 salida
modelo = nn.Sequential(
    nn.Linear(28, 128),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(32, 1)
)
criterio = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([5.0]))
optimizador = torch.optim.Adam(modelo.parameters(), lr=0.001)

print(modelo)

Sequential(
  (0): Linear(in_features=28, out_features=128, bias=True)
  (1): ReLU()
  (2): Dropout(p=0.2, inplace=False)
  (3): Linear(in_features=128, out_features=64, bias=True)
  (4): ReLU()
  (5): Dropout(p=0.2, inplace=False)
  (6): Linear(in_features=64, out_features=32, bias=True)
  (7): ReLU()
  (8): Dropout(p=0.2, inplace=False)
  (9): Linear(in_features=32, out_features=1, bias=True)
)


### 5.3. Entrenamiento

In [50]:
EPOCAS = 10

for epoca in range(1, EPOCAS + 1):

    # --- Entrenamiento ---
    modelo.train()
    perdida_train, aciertos_train, total_train = 0.0, 0, 0
    for x_lote, y_lote in train_loader:
        optimizador.zero_grad()               # limpiamos gradientes
        logits = modelo(x_lote)               # predicción
        perdida = criterio(logits, y_lote)    # error
        perdida.backward()                    # gradientes
        optimizador.step()                    # actualizamos pesos

        perdida_train += perdida.item() * len(x_lote)
        aciertos_train += ((torch.sigmoid(logits) >= 0.5).float() == y_lote).sum().item()
        total_train += len(x_lote)

    # --- Validación ---
    modelo.eval()
    perdida_val, aciertos_val, total_val = 0.0, 0, 0
    with torch.no_grad():
        for x_lote, y_lote in val_loader:
            logits = modelo(x_lote)
            perdida_val += criterio(logits, y_lote).item() * len(x_lote)
            aciertos_val += ((torch.sigmoid(logits) >= 0.5).float() == y_lote).sum().item()
            total_val += len(x_lote)

    print("Época %2d/%d | error: %.4f - accuracy: %.4f | val_error: %.4f - val_accuracy: %.4f"
          % (epoca, EPOCAS, perdida_train / total_train, aciertos_train / total_train,
             perdida_val / total_val, aciertos_val / total_val))

Época  1/10 | error: 0.0725 - accuracy: 0.9808 | val_error: 0.0135 - val_accuracy: 0.9991
Época  2/10 | error: 0.0114 - accuracy: 0.9993 | val_error: 0.0136 - val_accuracy: 0.9991
Época  3/10 | error: 0.0100 - accuracy: 0.9993 | val_error: 0.0137 - val_accuracy: 0.9992
Época  4/10 | error: 0.0092 - accuracy: 0.9994 | val_error: 0.0125 - val_accuracy: 0.9991
Época  5/10 | error: 0.0092 - accuracy: 0.9993 | val_error: 0.0119 - val_accuracy: 0.9991
Época  6/10 | error: 0.0084 - accuracy: 0.9993 | val_error: 0.0117 - val_accuracy: 0.9991
Época  7/10 | error: 0.0082 - accuracy: 0.9994 | val_error: 0.0120 - val_accuracy: 0.9989
Época  8/10 | error: 0.0073 - accuracy: 0.9994 | val_error: 0.0132 - val_accuracy: 0.9992
Época  9/10 | error: 0.0074 - accuracy: 0.9994 | val_error: 0.0110 - val_accuracy: 0.9990
Época 10/10 | error: 0.0070 - accuracy: 0.9994 | val_error: 0.0127 - val_accuracy: 0.9991


### 5.4. Métricas finales sobre el conjunto de prueba

In [51]:
# Predicción sobre el conjunto de prueba
modelo.eval()
with torch.no_grad():
    probabilidades = torch.sigmoid(modelo(X_test_t)).numpy().ravel()

y_pred = (probabilidades >= 0.5).astype(int)
y_real = y_test.values

print("Accuracy :", accuracy_score(y_real, y_pred))
print("Error    :", 1 - accuracy_score(y_real, y_pred))
print("Precisión:", precision_score(y_real, y_pred))
print("Recall   :", recall_score(y_real, y_pred))
print("F1 Score :", f1_score(y_real, y_pred))

print("\nMatriz de confusión:")
print(confusion_matrix(y_real, y_pred))

print("\n", classification_report(y_real, y_pred, target_names=["Legítima", "Fraude"]))

Accuracy : 0.9993328885923949
Error    : 0.0006671114076051143
Precisión: 0.7850467289719626
Recall   : 0.8484848484848485
F1 Score : 0.8155339805825242

Matriz de confusión:
[[56840    23]
 [   15    84]]

               precision    recall  f1-score   support

    Legítima       1.00      1.00      1.00     56863
      Fraude       0.79      0.85      0.82        99

    accuracy                           1.00     56962
   macro avg       0.89      0.92      0.91     56962
weighted avg       1.00      1.00      1.00     56962



### 5.5. Interpretación de los resultados

El conjunto de prueba contiene 56.962 transacciones, de las cuales solo 99 son fraudulentas (0,17%): por cada fraude hay unas 574 transacciones legítimas; esa proporción, heredada directamente de los datos de entrada, es la que explica el contraste entre el rendimiento global y el rendimiento por categoría.

Pongamos de ejemplo que un clasificador trivial que respondiera "legítima" en todos los casos obtendría un 99,83% de aciertos sin detectar un solo fraude, el margen real de mejora sobre esa línea base es de apenas 0,11 puntos porcentuales, de modo que un *accuracy* alto es aquí el punto de partida y no un logro.

**Lectura por categoría (matriz de confusión).**

|                   | Predicho legítima | Predicho fraude |
|-------------------|-------------------|-----------------|
| **Real legítima** | 56.840 (VN)       | 23 (FP)         |
| **Real fraude**   | 15 (FN)           | 84 (VP)         |

- *Recall* = 84/99 = **0,848** → el modelo detecta el 85% de los fraudes; se le escapan 15.
- Precisión = 84/107 = **0,785** → de cada 5 alertas emitidas, unas 4 son fraude real y 1 es falsa alarma.
- Tasa de falsos positivos = 23/56.863 = **0,040%** → se equivoca en 4 de cada 10.000 transacciones legítimas.

El parámetro `pos_weight=5` de `BCEWithLogitsLoss` multiplica por cinco el coste de no detectar un fraude, lo que desplaza deliberadamente el punto de operación hacia el *recall* a costa de la precisión, sin esa corrección, un MLP entrenado con un 0,17% de positivos tendería a predecir siempre "legítima", que es la estrategia que minimiza la pérdida global; el resultado obtenido (*recall* 0,848 > precisión 0,785) es por tanto el comportamiento buscado: en detección de fraude un falso negativo (una operación fraudulenta que pasa) suele ser mucho más costoso que un falso positivo (una operación legítima que se revisa manualmente). El umbral de 0,5 no es un valor sagrado; moverlo permite recorrer ese compromiso según el coste que se asigne a cada tipo de error.


**Conclusión.** El desequilibrio del conjunto de entrada no impide entrenar un clasificador útil, pero sí invalida el *accuracy* como criterio de evaluación y acota la precisión alcanzable sobre la clase minoritaria. La evaluación debe hacerse por categoría (matriz de confusión, precisión/*recall*/F1 de la clase Fraude).